# PromptFrame — Complete Usage Guide

A hands-on walkthrough of every feature, built to run top-to-bottom without any external files.

| Section | What you'll learn |
|---------|-------------------|
| **0** | Setup — all sample files created in-notebook |
| **1** | `PromptRegistry` — load YAML prompts across environments |
| **2** | `StructuredPromptBuilder` — compose prompts from components |
| **3** | `LLMBaseModel` + `LLMField` — structured schemas for LLM I/O |
| **4** | `SkillRegistry` — inject markdown skill files into prompts |
| **5** | `Parsers` — safely parse JSON from LLM responses |
| **6** | **Full pipeline** — all features wired together end-to-end |
| **7** | CLI quick reference |

> **Install:** `pip install promptframe`

---
## 0. Setup

Manual files are created

#### ── prompts/common/summarise.yaml ────────────────────────────────────────────

```YAML
version: 1.0
metadata:
  type: prompt
  name: summarise
  description: Text summarisation prompts (common baseline)
  tags: [nlp, summarise]
  project: demo
prompts:
  - pid: system_role
    description: System role for the summarisation assistant.
    prompt: |
      You are a professional summarisation assistant.
      Be concise, factual, and avoid unnecessary jargon.

  - pid: short_summary
    description: Summarise text in one sentence.
    input_variables: [text]
    prompt: |
      Summarise the following text in a single sentence:

      {text}

  - pid: bullet_summary
    description: Summarise as N bullet points.
    input_variables: [text, n]
    prompt: |
      Summarise the following text as exactly {n} bullet points:

      {text}
```

#### ── prompts/prod/summarise.yaml — tighter production override ─────────────────

```YAML
version: 1.0
metadata:
  type: prompt
  name: summarise
  description: Production summarisation prompts
  tags: [nlp, summarise]
  project: demo
prompts:
  - pid: system_role
    description: Stricter system role for production.
    prompt: |
      You are a senior professional editor.
      Respond ONLY with the requested summary. No preamble, no commentary.

  - pid: short_summary
    description: Summarise text in one sentence - PROD.
    input_variables: [text]
    prompt: |
      You are a professional editor. Summarise in one sentence, no jargon:

      {text}

  - pid: bullet_summary
    description: Summarise as N bullet points - PROD.
    input_variables: [text, n]
    prompt: |
      You are a professional editor. Produce exactly {n} concise bullet points:

      {text}
```

#### ── prompts/common/customer_prompts.yaml — model_prompt type ──────────────────

```YAML
version: 1.0
metadata:
  type: model_prompt
  name: customer_prompts
  description: Field-level LLM instructions for the Customer extraction model.
  tags: [customer, extraction]
  project: demo
prompts:
  - pid: name_prompt
    model_attribute_id: customer_name
    input_instruction: |
      The raw input may contain a messy, abbreviated, or informal name.
      Handle nicknames, initials, and mixed casing.
    output_instruction: |
      Return the full name in Title Case, e.g. 'John Smith'.
      Do not include titles (Mr/Ms/Dr) unless explicitly part of the name.

  - pid: email_prompt
    model_attribute_id: customer_email
    input_instruction: |
      The input may contain a raw or malformed email address.
    output_instruction: |
      Return a valid lowercase email address. If none is found, return null.

  - pid: tier_prompt
    model_attribute_id: customer_tier
    input_instruction: |
      Look for clues about the customer's value tier in the text.
    output_instruction: |
      Return one of: 'bronze', 'silver', 'gold', 'platinum'.
      Default to 'bronze' if no clear signal.
```

#### ── prompts/skills/python-expert/SKILL.md ────────────────────────────────────────────

```YAML
---
name: Python Expert
description: Best practices for production-quality Python code.
tags:
  - python
  - backend
  - engineering
version: "1.0"
---
```

## Code Style

Follow PEP 8. Use type hints on all public functions and class attributes.
Prefer `pathlib.Path` over `os.path`. Use f-strings, not `.format()` or `%`.

## Error Handling

Raise specific named exceptions. Never use bare `except:` clauses.
Log errors with full context using the `logging` module.

## Testing

Write tests with `pytest`. Aim for 80%+ coverage on all business logic.
Use `pytest.mark.parametrize` for table-driven tests.

## Performance

Profile before optimising. Use `cProfile` or `py-spy`.
Prefer generators over lists for large data streams.


#### ── skills/data-analysis/SKILL.md ────────────────────────────────────────────

```YAML
---
name: Data Analysis
description: Guidelines for reliable exploratory data analysis.
tags:
  - data
  - pandas
  - analytics
version: "1.0"
---
```
## Data Loading

Always inspect shape, dtypes, and null counts before any analysis.
Document all data sources and their refresh cadence.

## Data Cleaning

Be explicit about how nulls are handled (drop, fill, flag).
Validate column ranges and categorical values before transforming.
Never modify the original DataFrame in-place without a backup.

## Visualisation

Use matplotlib or seaborn. Label all axes with units. Include a title.
Use colorblind-friendly palettes.

---
## 1. PromptRegistry — YAML-based Prompt Management

The registry resolves the right YAML file by environment priority: `environment/` → `common/` → `base/`.

In [1]:
PROMPTS = "prompts"
SKILLS = "prompts/skills"

In [2]:
from promptframe import PromptRegistry

reg_common = PromptRegistry(base=str(PROMPTS), common="common")
reg_prod   = PromptRegistry(base=str(PROMPTS), environment="prod", common="common")

p_common = reg_common.load_prompt("summarise")
p_prod   = reg_prod.load_prompt("summarise")

print("Common:", repr(p_common))
print("Prod:  ", repr(p_prod))

Common: PromptYAML(name='summarise', prompts=['system_role', 'short_summary', 'bullet_summary'])
Prod:   PromptYAML(name='summarise', prompts=['system_role', 'short_summary', 'bullet_summary'])


In [3]:
# Attribute access by pid
print("=== common/short_summary ===")
print(p_common.short_summary.prompt)

print("\n=== prod/short_summary (overridden) ===")
print(p_prod.short_summary.prompt)

=== common/short_summary ===
Summarise the following text in a single sentence:

{text}


=== prod/short_summary (overridden) ===
You are a professional editor. Summarise in one sentence, no jargon:

{text}



In [4]:
# .format(**kwargs) fills {placeholders}
TEXT = (
    "The James Webb Space Telescope has captured unprecedented images of "
    "exoplanet atmospheres, revealing carbon dioxide and water vapour "
    "in systems hundreds of light-years away."
)

print(p_prod.short_summary.format(text=TEXT))
print()
print(p_prod.bullet_summary.format(text=TEXT, n=3))

You are a professional editor. Summarise in one sentence, no jargon:

The James Webb Space Telescope has captured unprecedented images of exoplanet atmospheres, revealing carbon dioxide and water vapour in systems hundreds of light-years away.


You are a professional editor. Produce exactly 3 concise bullet points:

The James Webb Space Telescope has captured unprecedented images of exoplanet atmospheres, revealing carbon dioxide and water vapour in systems hundreds of light-years away.


In [5]:
# Dict access + safe .get()
print("All pids:", list(p_prod.prompt_dict.keys()))
print("Files:   ", reg_prod.list_prompts())
print("Missing: ", p_prod.get("nonexistent_pid"))  # returns None, no exception

All pids: ['system_role', 'short_summary', 'bullet_summary']
Files:    ['summarise.yaml', 'customer_prompts.yaml']
Missing:  None


---
## 2. StructuredPromptBuilder — Composing Prompts

Chain components with `>>`. The builder renders them into a single string joined by blank lines.

In [6]:
from promptframe.builder import StructuredPromptBuilder
from promptframe.components.basic import (
    SimplePromptComponent,
    PromptSectionComponent,
    InputComponent,
    ConditionalPromptComponent,
    TemplatePromptComponent,
    SequentialPromptComponent,
    SkillComponent,
)

In [7]:
# ── SimplePromptComponent — plain string or Prompt object ────────────────────
builder = (
    StructuredPromptBuilder()
    >> SimplePromptComponent(p_prod.system_role)           # Prompt object from registry
    >> SimplePromptComponent("Summarise this: {text}")     # plain string with placeholder
)

print(builder.build({"text": "AI is transforming every industry."}))
print(f"\n>>Total components: {len(builder)}")

You are a senior professional editor.
Respond ONLY with the requested summary. No preamble, no commentary.


Summarise this: AI is transforming every industry.

>>Total components: 2


In [8]:
# ── PromptSectionComponent — header + bullet list ────────────────────────────
constraints = PromptSectionComponent(
    requirement=[
        "Respond in the same language as the input.",
        "Never fabricate facts — say 'unknown' if unsure.",
        "Keep the response under 150 words.",
    ],
    header="Constraints:",
)
print(constraints.render())

Constraints:
- Respond in the same language as the input.
- Never fabricate facts — say 'unknown' if unsure.
- Keep the response under 150 words.


In [9]:
# ── ConditionalPromptComponent — sections that only appear when a flag is set ─
json_gate    = ConditionalPromptComponent(
    component=SimplePromptComponent("MUST respond with a valid JSON object. No prose."),
    condition_key="json_mode",
)
verbose_gate = ConditionalPromptComponent(
    component=SimplePromptComponent("Explain your reasoning step-by-step first."),
    condition_key="verbose",
)

flexible = (
    StructuredPromptBuilder()
    >> SimplePromptComponent("You are an intelligent assistant.")
    >> json_gate
    >> verbose_gate
    >> InputComponent(header="Question:", template="{question}")
)

print("--- json=True, verbose=False ---")
print(flexible.build({"json_mode": True,  "verbose": False, "question": "What is 2+2?"}))

print("\n--- json=False, verbose=True ---")
print(flexible.build({"json_mode": False, "verbose": True,  "question": "What is 2+2?"}))

--- json=True, verbose=False ---
You are an intelligent assistant.

MUST respond with a valid JSON object. No prose.

Question:
What is 2+2?

--- json=False, verbose=True ---
You are an intelligent assistant.

Explain your reasoning step-by-step first.

Question:
What is 2+2?


In [10]:
# ── TemplatePromptComponent — named slots composed from other components ──────
chat_template = TemplatePromptComponent(
    template="{system}\n\n{rules}\n\n{input_block}",
    components={
        "system": SimplePromptComponent(p_prod.system_role),
        "rules": constraints,
        "input_block": InputComponent(header="Text:", template="<doc>{text}</doc>"),
    },
)
print(chat_template.render({"text": "Quantum computing may break current encryption."}))

You are a senior professional editor.
Respond ONLY with the requested summary. No preamble, no commentary.


Constraints:
- Respond in the same language as the input.
- Never fabricate facts — say 'unknown' if unsure.
- Keep the response under 150 words.

Text:
<doc>Quantum computing may break current encryption.</doc>


In [11]:
# ── SequentialPromptComponent — with render
seq = (
    StructuredPromptBuilder()
    >> SimplePromptComponent("Part A: {a}")
    >> SimplePromptComponent("Part B: {b}")
    >> SimplePromptComponent("Part C: {c}")
)
print(seq.build({"a": "intro", "b": "body", "c": "conclusion"}))

Part A: intro

Part B: body

Part C: conclusion


---
## 3. LLMBaseModel + LLMField — Structured I/O Schemas

Define your model once. Get both the *input schema* (what you tell the LLM about your data) and *output schema* (how it should format its response) automatically.

In [12]:
from typing import List, Literal, Optional
from promptframe.llm_base_model import LLMBaseModel
from promptframe.fields import LLMField


class Address(LLMBaseModel):
    street:  str           = LLMField(...,  description="Full street address line.")
    city:    str           = LLMField(...,  description="City or town name.")
    zip:     str           = LLMField(...,  description="Postal / ZIP code.")
    country: Optional[str] = LLMField(None, description="ISO 2-letter country code, e.g. 'US'.")


class Customer(LLMBaseModel):
    name: str = LLMField(
        ...,
        description="Customer full name.",
        model_attribute_id="customer_name",       # links to customer_prompts.yaml
        output_instruction="Return in Title Case.",
    )
    email: Optional[str] = LLMField(
        None,
        description="Customer email address.",
        model_attribute_id="customer_email",
        output_instruction="Lowercase valid email, or null if absent.",
    )
    age:  Optional[int] = LLMField(None, description="Customer age in years.")
    tier: Literal["bronze", "silver", "gold", "platinum"] = LLMField(
        "bronze",
        description="Customer loyalty tier.",
        model_attribute_id="customer_tier",
    )
    address: Address   = LLMField(...,                description="Customer shipping address.")
    tags:    List[str] = LLMField(default_factory=list, description="Freeform descriptive tags.")


print("Models defined: Customer, Address")

Models defined: Customer, Address


In [13]:
# ── Input instructions (describe YOUR data to the LLM) ───────────────────────
print(Customer.get_input_instructions())


Here is the input data schema with embedded field instructions and metadata:
<input_schema>{
  "name": {
    "instruction": "Customer full name."
  },
  "email": {
    "instruction": "Customer email address."
  },
  "age": {
    "instruction": "Customer age in years."
  },
  "tier": {
    "instruction": "Customer loyalty tier."
  },
  "address": {
    "instruction": "Customer shipping address.",
    "fields": {
      "street": {
        "instruction": "Full street address line."
      },
      "city": {
        "instruction": "City or town name."
      },
      "zip": {
        "instruction": "Postal / ZIP code."
      },
      "country": {
        "instruction": "ISO 2-letter country code, e.g. 'US'."
      }
    }
  },
  "tags": {
    "instruction": "Freeform descriptive tags."
  }
}</input_schema>



In [14]:
import json
# ── Output format instructions — get_dict=True for the raw schema ─────────────
schema = Customer.get_format_instructions(get_dict=True)
print(json.dumps(schema, indent=2))

{
  "properties": {
    "name": {
      "output_instruction": "Return in Title Case.",
      "title": "Name",
      "type": "string"
    },
    "email": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "output_instruction": "Lowercase valid email, or null if absent.",
      "title": "Email"
    },
    "age": {
      "anyOf": [
        {
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Age",
      "output_instruction": "Customer age in years."
    },
    "tier": {
      "default": "bronze",
      "enum": [
        "bronze",
        "silver",
        "gold",
        "platinum"
      ],
      "title": "Tier",
      "type": "string",
      "output_instruction": "Customer loyalty tier."
    },
    "address": {
      "properties": {
        "street": {
          "title": "Street",
          "type": "string",
  

In [15]:
# ── Ignore specific fields (e.g. fields you populate server-side) ─────────────
partial = Customer.get_input_instructions(
    get_dict=True,
    ignore=("age", "address.country"),
)
print(json.dumps(partial, indent=2))

{
  "name": {
    "instruction": "Customer full name."
  },
  "email": {
    "instruction": "Customer email address."
  },
  "tier": {
    "instruction": "Customer loyalty tier."
  },
  "address": {
    "instruction": "Customer shipping address.",
    "fields": {
      "street": {
        "instruction": "Full street address line."
      },
      "city": {
        "instruction": "City or town name."
      },
      "zip": {
        "instruction": "Postal / ZIP code."
      }
    }
  },
  "tags": {
    "instruction": "Freeform descriptive tags."
  }
}


In [16]:
# ── Bind YAML model_prompts — inject richer per-field instructions ─────────────
mp = reg_common.load_model_prompt("customer_prompts")
print("Binding keys:", list(mp.prompt_model_dict.keys()))

# Input with YAML input_instructions overriding field descriptions
print()
print(Customer.get_input_instructions_with_prompt(prompt_model_dict=mp.prompt_model_dict))

Binding keys: ['customer_name', 'customer_email', 'customer_tier']


Here is the input data schema with embedded field instructions and metadata:
<input_schema>{
  "name": {
    "instruction": "The raw input may contain a messy, abbreviated, or informal name.\nHandle nicknames, initials, and mixed casing.\n"
  },
  "email": {
    "instruction": "The input may contain a raw or malformed email address.\n"
  },
  "age": {
    "instruction": "Customer age in years."
  },
  "tier": {
    "instruction": "Look for clues about the customer's value tier in the text.\n"
  },
  "address": {
    "instruction": "Customer shipping address.",
    "fields": {
      "street": {
        "instruction": "Full street address line."
      },
      "city": {
        "instruction": "City or town name."
      },
      "zip": {
        "instruction": "Postal / ZIP code."
      },
      "country": {
        "instruction": "ISO 2-letter country code, e.g. 'US'."
      }
    }
  },
  "tags": {
    "instruction": "

In [17]:
# Output schema with YAML output_instructions injected
out = Customer.get_format_instructions_with_prompt(
    prompt_model_dict=mp.prompt_model_dict,
    get_dict=True,
)
# Show just the properties section for brevity
print(json.dumps(out.get("properties", out), indent=2))

{
  "name": {
    "output_instruction": "Return the full name in Title Case, e.g. 'John Smith'.\nDo not include titles (Mr/Ms/Dr) unless explicitly part of the name.\n",
    "title": "Name",
    "type": "string"
  },
  "email": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "output_instruction": "Return a valid lowercase email address. If none is found, return null.\n",
    "title": "Email"
  },
  "age": {
    "anyOf": [
      {
        "type": "integer"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "title": "Age",
    "output_instruction": "Customer age in years."
  },
  "tier": {
    "default": "bronze",
    "enum": [
      "bronze",
      "silver",
      "gold",
      "platinum"
    ],
    "title": "Tier",
    "type": "string",
    "output_instruction": "Return one of: 'bronze', 'silver', 'gold', 'platinum'.\nDefault to 'bronze' if no clear signal."
  },
  "address": 

In [18]:
# ── get_llm_schema — input + output in one call ───────────────────────────────
full = Customer.get_llm_schema(prompt_model_dict=mp.prompt_model_dict, get_dict=True)
print("Keys:", list(full.keys()))

# force=True busts the cache — useful after redefining the model
_ = Customer.get_input_instructions(force=True)
print("Cache cleared OK")

Keys: ['input', 'output']
Cache cleared OK


---
## 4. SkillRegistry — Markdown Skill Files

Skills are rich markdown documents — coding standards, decision frameworks, best practices — injected verbatim into prompts as context.

In [19]:
from promptframe import SkillRegistry

skill_reg = SkillRegistry(str(SKILLS))

# ── List all discovered skills ────────────────────────────────────────────────
print("Discovered skills:")
for s in skill_reg.list():
    print(f"  [{s['key']}]  {s['description']}")

Discovered skills:
  [data-analysis]  Guidelines for reliable exploratory data analysis.
  [python-expert]  Best practices for production-quality Python code.


In [20]:
# ── Load a skill and inspect its sections ─────────────────────────────────────
py_skill = skill_reg.get("python-expert")

print(repr(py_skill))
print("Sections:", list(py_skill.sections.keys()))
print()
print("--- Error Handling ---")
print(py_skill.get_section("Error Handling"))

Skill(name='python-expert', tags=['python', 'backend', 'engineering'], sections=['Code Style', 'Error Handling', 'Testing', 'Performance'])
Sections: ['Code Style', 'Error Handling', 'Testing', 'Performance']

--- Error Handling ---
Raise specific named exceptions. Never use bare `except:` clauses.
Log errors with full context using the `logging` module.


In [21]:
# ── Render: full skill vs selected sections ───────────────────────────────────
print("=== Selected sections only ===")
print(py_skill.render(sections=["Code Style", "Testing"]))

print("\n=== With XML wrapper ===")
da_skill = skill_reg.get("data-analysis")
print(da_skill.render(sections=["Data Cleaning"]))

=== Selected sections only ===
## python-expert

## Code Style
Follow PEP 8. Use type hints on all public functions and class attributes.
Prefer `pathlib.Path` over `os.path`. Use f-strings, not `.format()` or `%`.

## Testing
Write tests with `pytest`. Aim for 80%+ coverage on all business logic.
Use `pytest.mark.parametrize` for table-driven tests.

=== With XML wrapper ===
## data-analysis

## Data Cleaning
Be explicit about how nulls are handled (drop, fill, flag).
Validate column ranges and categorical values before transforming.
Never modify the original DataFrame in-place without a backup.


In [22]:
# ── SkillComponent — inject a skill into a builder ────────────────────────────
code_prompt = (
    StructuredPromptBuilder()
    >> SimplePromptComponent("You are a senior Python engineer.")
    >> SkillComponent(
           py_skill,
           sections=["Code Style", "Error Handling"],
           wrapper="<engineering_guidelines>\n{skill}\n</engineering_guidelines>",
       )
    >> InputComponent(header="Your task:", template="{task}")
).build({"task": "Write a robust CSV reader that handles missing values and encoding errors."})

print(code_prompt)

You are a senior Python engineer.

<engineering_guidelines>
## python-expert

## Code Style
Follow PEP 8. Use type hints on all public functions and class attributes.
Prefer `pathlib.Path` over `os.path`. Use f-strings, not `.format()` or `%`.

## Error Handling
Raise specific named exceptions. Never use bare `except:` clauses.
Log errors with full context using the `logging` module.
</engineering_guidelines>

Your task:
Write a robust CSV reader that handles missing values and encoding errors.


In [23]:
# ── Load skill via PromptRegistry.load_skill ──────────────────────────────────
# Useful when skills live alongside prompts in the same registry base
reg_all  = PromptRegistry(base=str(PROMPTS))
da_skill = reg_all.load_skill("skills/data-analysis")
print(repr(da_skill))

# load_all() caches every skill at once
all_skills = skill_reg.load_all()
print("All loaded:", list(all_skills.keys()))

Skill(name='data-analysis', tags=['data', 'pandas', 'analytics'], sections=['Data Loading', 'Data Cleaning', 'Visualisation'])
All loaded: ['python-expert', 'data-analysis']


---
## 5. Parsers — Extract JSON from LLM Responses

LLMs often wrap JSON in markdown fences, add prose, or truncate. These parsers handle all of it.

In [24]:
from promptframe.parsers import json_parser, parse_partial_json, parse_json_markdown
from promptframe.exceptions import OutputParsingError

# ── Clean JSON string ─────────────────────────────────────────────────────────
print(json_parser('{"name": "Alice", "age": 30}'))

{'name': 'Alice', 'age': 30}


In [25]:
# ── Markdown-fenced JSON — most common LLM output format ─────────────────────
fenced = """
Sure! Here is the extracted data:

```json
{"name": "Bob Smith", "tier": "gold", "email": "bob@example.com"}
```

Let me know if you need anything else!
"""
print(parse_json_markdown(fenced))

{'name': 'Bob Smith', 'tier': 'gold', 'email': 'bob@example.com'}


In [26]:
# ── Partial / truncated JSON — LLM hit a token limit ─────────────────────────
truncated = '{"name": "Carol", "tags": ["vip", "returning'
print("Recovered:", parse_partial_json(truncated))

Recovered: {'name': 'Carol', 'tags': ['vip', 'returning']}


In [27]:
# ── Error handling ────────────────────────────────────────────────────────────
try:
    json_parser("This is plain prose, not JSON.")
except OutputParsingError as e:
    print(f"Caught OutputParsingError: {e.message}")

Caught OutputParsingError: Failed to parse the LLM response output.


---
## 6. Full Pipeline — All Features Wired Together

A realistic **customer data extraction** pipeline combining every feature:

```
PromptRegistry  →  system role + summarise prompts (prod override)
    +
model_prompt YAML  →  per-field input/output instructions
    +
SkillRegistry  →  data-analysis skill injected as context
    +
StructuredPromptBuilder  →  assembles all pieces into one prompt
    +
Parsers  →  parse the LLM JSON response
    +
Customer(LLMBaseModel)  →  validates and types the result
```

In [28]:
# ── Step 1: All registries ─────────────────────────────────────────────────────
registry  = PromptRegistry(base=str(PROMPTS), environment="prod", common="common")
skill_reg = SkillRegistry(str(SKILLS))

# ── Step 2: Load prompts ───────────────────────────────────────────────────────
summarise     = registry.load_prompt("summarise")           # prod override
model_prompts = registry.load_model_prompt("customer_prompts")  # common
system_role   = summarise.system_role

# ── Step 3: Load skill ─────────────────────────────────────────────────────────
da_skill = skill_reg.get("data-analysis")

print("Registries ready")
print("System role pid:", system_role.pid)
print("Model prompt pids:", list(model_prompts.prompt_dict.keys()))
print("Skill sections:", list(da_skill.sections.keys()))

Registries ready
System role pid: system_role
Model prompt pids: ['name_prompt', 'email_prompt', 'tier_prompt']
Skill sections: ['Data Loading', 'Data Cleaning', 'Visualisation']


In [29]:
# ── Step 4: Generate schemas with YAML prompt overrides ───────────────────────
pd = model_prompts.prompt_model_dict

input_schema  = Customer.get_input_instructions_with_prompt(prompt_model_dict=pd)
output_schema = Customer.get_format_instructions_with_prompt(prompt_model_dict=pd)

print("Schemas generated")
print("Input schema preview:")
print(input_schema[:350], "...")

Schemas generated
Input schema preview:

Here is the input data schema with embedded field instructions and metadata:
<input_schema>{
  "name": {
    "instruction": "The raw input may contain a messy, abbreviated, or informal name.\nHandle nicknames, initials, and mixed casing.\n"
  },
  "email": {
    "instruction": "The input may contain a raw or malformed email address.\n"
  },
  "age ...


In [31]:
# ── Step 5: Assemble the full prompt ──────────────────────────────────────────
RAW_TEXT = (
    "Name: alice smith  |  Email: ALICE@Example.COM  |  Age: 34  |  "
    "Address: 42 Elm Street, Boston, 02101, US  |  "
    "Been with us since 2019, always buys premium — definitely gold tier."
)

extraction_prompt = (
    StructuredPromptBuilder()

    # System role loaded from prod YAML
    >> SimplePromptComponent(system_role)

    # Task description
    >> SimplePromptComponent(
        "Your task is to extract structured customer information from raw text."
    )

    # Data quality guidelines from the skill (only the relevant section)
    >> SkillComponent(
        da_skill,
        sections=["Data Cleaning"],
        include_name=False,
        wrapper="<data_quality_guidelines>\n{skill}\n</data_quality_guidelines>",
    )

    # Field-level input instructions (with YAML overrides)
    >> SimplePromptComponent(input_schema)

    # Output format instructions (with YAML output_instructions)
    >> SimplePromptComponent(output_schema)

    # Optional strict-mode reminder — only shown when flag is set
    >> ConditionalPromptComponent(
        component=SimplePromptComponent(
            "STRICT MODE: every field is required. Use null for any absent optional values."
        ),
        condition_key="strict_mode",
    )

    # Raw input wrapped in XML tags
    >> InputComponent(
        header="Extract structured data from the raw text below:",
        template="<raw_text>\n{raw_text}\n</raw_text>",
    )
).build({"strict_mode": True, "raw_text": RAW_TEXT})

print(extraction_prompt)

You are a senior professional editor.
Respond ONLY with the requested summary. No preamble, no commentary.


Your task is to extract structured customer information from raw text.

<data_quality_guidelines>
## Data Cleaning
Be explicit about how nulls are handled (drop, fill, flag).
Validate column ranges and categorical values before transforming.
Never modify the original DataFrame in-place without a backup.
</data_quality_guidelines>


Here is the input data schema with embedded field instructions and metadata:
<input_schema>{
  "name": {
    "instruction": "The raw input may contain a messy, abbreviated, or informal name.\nHandle nicknames, initials, and mixed casing.\n"
  },
  "email": {
    "instruction": "The input may contain a raw or malformed email address.\n"
  },
  "age": {
    "instruction": "Customer age in years."
  },
  "tier": {
    "instruction": "Look for clues about the customer's value tier in the text.\n"
  },
  "address": {
    "instruction": "Customer shipping a

In [32]:
# ── Step 6: Parse the LLM response and validate ───────────────────────────────
# Simulating a real LLM response (markdown-fenced JSON)
mock_llm_response = """
```json
{
  "name": "Alice Smith",
  "email": "alice@example.com",
  "age": 34,
  "tier": "gold",
  "address": {
    "street": "42 Elm Street",
    "city": "Boston",
    "zip": "02101",
    "country": "US"
  },
  "tags": ["premium", "long-term", "gold-tier"]
}
```
"""

parsed   = json_parser(mock_llm_response)
customer = Customer(**parsed)

print("Parsed and validated successfully")
print()
print(f"  Name:    {customer.name}")
print(f"  Email:   {customer.email}")
print(f"  Tier:    {customer.tier}")
print(f"  Age:     {customer.age}")
print(f"  City:    {customer.address.city}")
print(f"  Country: {customer.address.country}")
print(f"  Tags:    {customer.tags}")

Parsed and validated successfully

  Name:    Alice Smith
  Email:   alice@example.com
  Tier:    gold
  Age:     34
  City:    Boston
  Country: US
  Tags:    ['premium', 'long-term', 'gold-tier']


In [33]:
# ── Step 7: Export the validated result ───────────────────────────────────────
print(json.dumps(customer.model_dump(), indent=2))

{
  "name": "Alice Smith",
  "email": "alice@example.com",
  "age": 34,
  "tier": "gold",
  "address": {
    "street": "42 Elm Street",
    "city": "Boston",
    "zip": "02101",
    "country": "US"
  },
  "tags": [
    "premium",
    "long-term",
    "gold-tier"
  ]
}


In [34]:
# ── Bonus: preview the full pipeline component-by-component ───────────────────
(
    StructuredPromptBuilder()
    >> SimplePromptComponent(system_role)
    >> SimplePromptComponent("Extract structured customer information from raw text.")
    >> SkillComponent(da_skill, sections=["Data Cleaning"], include_name=False)
    >> SimplePromptComponent(input_schema)
    >> SimplePromptComponent(output_schema)
    >> ConditionalPromptComponent(
           component=SimplePromptComponent("STRICT MODE active."),
           condition_key="strict_mode",
       )
    >> InputComponent(header="Raw text:", template="<raw_text>{raw_text}</raw_text>")
).preview({"strict_mode": True, "raw_text": "alice smith, alice@example.com, gold"})

🔍 Prompt Preview
[0] SimplePromptComponent
------------------------------
You are a senior professional editor.
Respond ONLY with the requested summary. No preamble, no commentary.


[1] SimplePromptComponent
------------------------------
Extract structured customer information from raw text.

[2] SkillComponent
------------------------------
## Data Cleaning
Be explicit about how nulls are handled (drop, fill, flag).
Validate column ranges and categorical values before transforming.
Never modify the original DataFrame in-place without a backup.

[3] SimplePromptComponent
------------------------------

Here is the input data schema with embedded field instructions and metadata:
<input_schema>{
  "name": {
    "instruction": "The raw input may contain a messy, abbreviated, or informal name.\nHandle nicknames, initials, and mixed casing.\n"
  },
  "email": {
    "instruction": "The input may contain a raw or malformed email address.\n"
  },
  "age": {
    "instruction": "Customer age i

---
## 7. CLI Quick Reference

```bash
# ── Prompt commands ────────────────────────────────────────────────────────────
promptframe scaffold prompts/ --example        # scaffold common/dev/test/prod dirs
promptframe init regular prompts/common/my.yaml
promptframe init model   prompts/common/my_model.yaml

promptframe list     prompts/                  # list all YAML files found
promptframe validate prompts/                  # validate structure
promptframe inspect  prompts/common/my.yaml    # metadata + prompt IDs
promptframe render   prompts/common/my.yaml short_summary
promptframe lint     prompts/                  # warn on missing descriptions
promptframe export   prompts/common/my.yaml --format json -o out.json
promptframe diff     prompts/common/v1.yaml prompts/common/v2.yaml

# ── Skill commands ─────────────────────────────────────────────────────────────
promptframe skill init     python-expert --path skills/
promptframe skill list     skills/
promptframe skill inspect  python-expert --path skills/
promptframe skill render   python-expert --path skills/ --section "Code Style"
promptframe skill validate skills/
promptframe skill lint     skills/
promptframe skill search   python --path skills/
promptframe skill diff     skills/v1/SKILL.md skills/v2/SKILL.md
```